In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()  # This loads the environment variables from .env

True

In [3]:
## Langsmith Tracking And Tracing
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]="true" #tells LangChain to turn on its tracing system (v2).


# Tracing means keeping a step-by-step record of how a program runs.›

In [8]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model="o1-mini")
response = llm.invoke("Who is Tajamul Khan a data scientist")

print(response)

content="As of my knowledge cutoff in October 2023, there isn't widely available public information about a data scientist named Tajamul Khan. It's possible that he is a professional in the field who hasn't gained significant media attention or recognition on major platforms. If you're looking for specific information about his work, contributions, or background, I recommend checking professional networking sites like LinkedIn, academic publications, or any organizations he might be affiliated with for the most accurate and up-to-date information.\n\nIf you have more context or details about his work or where he's based, feel free to share, and I can try to provide more targeted assistance based on that information." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 395, 'prompt_tokens': 16, 'total_tokens': 411, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens

In [21]:
# Chaining : We need prompt | Model | Outputparser

# Prompt

from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are an expert AI Engineer. Provide me answer based on the question"),
        ("user","{input}")
    ]
)
prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert AI Engineer. Provide me answer based on the question'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [20]:
# Model 
from langchain_groq import ChatGroq
model=ChatGroq(model="gemma2-9b-it")
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x1075ee660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1096412b0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [22]:
#Output Parser

from langchain_core.output_parsers import StrOutputParser  
output_parser = StrOutputParser()  

In [28]:
# Chaining 

chain = prompt | model | output_parser  
response = chain.invoke({"input":"Can you tell me about Langsmith in one line"})  
print(response)

Langsmith is an open-source toolkit for building and deploying large language models (LLMs) more efficiently. 



In [34]:
# Prompt - just make sure to add in instructions
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are an expert AI Engineer. Provide me answer in XML and json both format"),
        ("user","{input}")
    ]
)
# Model 
from langchain_groq import ChatGroq
model=ChatGroq(model="gemma2-9b-it")

# Output
from langchain_core.output_parsers import StrOutputParser  
output_parser = StrOutputParser()

# Chaining
chain = prompt|model|output_parser  
response = chain.invoke({"input":"Can you tell me about Langsmith in one line"})  
print(response)

```xml
<response>
  <description>Langsmith is an open-source platform for building and deploying AI agents.</description>
</response>
```

```json
{
  "description": "Langsmith is an open-source platform for building and deploying AI agents."
}
```



### Assignment :

Create a simple assistant that uses any LLM and should be pydantic, when we ask about any product it should give you two information product Name, product details tentative price in USD (integer). use chat Prompt Template.

In [76]:
import os
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser  
output_parser = StrOutputParser()

In [90]:
# Inititate Pydantic Class for outputparser
class Product_details(BaseModel):
    product_name: str
    product_configurations: str
    tentative_price_usd: int

In [91]:
from langchain_core.output_parsers import JsonOutputParser

# Initialize Output Parser with Pydantic schema
output_parser = JsonOutputParser(pydantic_object=Product_details)

In [92]:
# Initite Model
model=ChatOpenAI(temperature=0.7)

In [93]:
# Initiate Chat Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that can answer questions about products 
    and must provide information strictly in the format specified."""),
    ("user", "Give the details of {input}.\n{format_instructions}") #format instructions is very imp here
])

In [ ]:
# Chain
chain = prompt | model | output_parser

# Invoke with dict input (matches {query} and {input} in your prompt)
response = chain.invoke({
    "query": "Give details",
    "input": "iphone 13",
    "format_instructions": output_parser.get_format_instructions()
})

print(response)

{'product_name': 'iPhone 13', 'product_configurations': 'Various configurations available with different storage capacities and colors.', 'tentative_price_usd': 799}
